# 00 — Inventario, filtro de Cali y trazabilidad

**Proyecto:** Vigía Cali — sistema auditable de vigilancia temporal de la criminalidad reportada para la planeación institucional.

**Autores:** completar manualmente antes de la entrega.

## Propósito

Auditar las ocho fuentes SIEDCO y los archivos poblacionales DANE antes
de cualquier consolidación. Cada fuente se lee, valida y filtra por
código oficial de Cali de forma independiente.

**Reglas no negociables**

- Los archivos originales de Drive son de solo lectura.
- Una fila SIEDCO es un agregado, no un evento individual.
- Los totales se obtienen sumando `cantidad`.
- Se acepta `cod_muni = 76001` o `codigo_dane = 76001000`.
- `municipio` solo sirve como validación secundaria.
- Los resultados se guardan en
  `MyDrive/datav3/project_diplodata_outputs/eda_01_05_v1`.


## 1. Configuración reproducible

Esta celda monta Drive y declara entradas y salidas. Reejecutar con el
mismo identificador reemplaza únicamente productos derivados, nunca
las fuentes originales.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

DATAV3_ROOT = Path("/content/drive/MyDrive/datav3")
SIEDCO_INPUT = DATAV3_ROOT / "A1 - SIEDCO" / "datos_criminalidad_cali"
PROJECT_OUTPUT = DATAV3_ROOT / "project_diplodata_outputs" / "eda_01_05_v1"
LANDING = PROJECT_OUTPUT / "landing"
TRUSTED = PROJECT_OUTPUT / "trusted"
SURFACE = PROJECT_OUTPUT / "surface"
AUDIT = PROJECT_OUTPUT / "audit"
REPORTS = PROJECT_OUTPUT / "reportes"

for directory in (LANDING, TRUSTED, SURFACE, AUDIT, REPORTS):
    directory.mkdir(parents=True, exist_ok=True)

PERIODO_INICIO = 2018
PERIODO_FIN = 2025

import hashlib

EXPECTED_SIEDCO_FILES = 8
SIEDCO_LANDING = LANDING / "siedco"
DANE_LANDING = LANDING / "dane"
SIEDCO_LANDING.mkdir(exist_ok=True)
DANE_LANDING.mkdir(exist_ok=True)

print("Entrada SIEDCO:", SIEDCO_INPUT)
print("Búsqueda DANE:", DATAV3_ROOT)
print("Salida del proyecto:", PROJECT_OUTPUT)


## 2. Funciones de lectura y normalización

Se prueban codificaciones y separadores comunes. Las columnas se
normalizan sin perder el nombre original en el registro de auditoría.
Las fechas se convierten fuente por fuente; los fallos quedan contados.


In [ ]:
def normalize_name(value):
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(c for c in text if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")


def file_sha256(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(block_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_csv_robust(path):
    attempts = []
    for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        for separator in (None, ",", ";", "\t", "|"):
            try:
                options = {"encoding": encoding, "low_memory": False}
                if separator is None:
                    options.update({"sep": None, "engine": "python"})
                else:
                    options["sep"] = separator
                frame = pd.read_csv(path, **options)
                if frame.shape[1] > 1:
                    return frame, encoding, separator or "autodetectado"
            except Exception as exc:
                attempts.append(f"{encoding}/{separator}: {exc}")
    raise ValueError("No fue posible leer el CSV. " + " | ".join(attempts[-4:]))


def parse_dates_robust(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        return pd.to_datetime(series, errors="coerce")
    parsed = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")
    text = series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    for date_format in ("%d/%m/%Y", "%Y-%m-%d", "%d-%m-%Y", "%Y/%m/%d", "%d.%m.%Y"):
        pending = parsed.isna() & text.notna()
        parsed.loc[pending] = pd.to_datetime(
            text.loc[pending], format=date_format, errors="coerce"
        )
    pending = parsed.isna() & text.notna()
    if pending.any():
        try:
            parsed.loc[pending] = pd.to_datetime(
                text.loc[pending], format="mixed", dayfirst=True, errors="coerce"
            )
        except (TypeError, ValueError):
            parsed.loc[pending] = pd.to_datetime(
                text.loc[pending], dayfirst=True, errors="coerce"
            )
    numeric = pd.to_numeric(series, errors="coerce")
    excel_serial = parsed.isna() & numeric.between(20000, 80000)
    parsed.loc[excel_serial] = pd.to_datetime(
        numeric.loc[excel_serial],
        unit="D",
        origin="1899-12-30",
        errors="coerce",
    )
    return parsed


def normalize_code(series):
    return series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)


def infer_source_type(filename):
    name = normalize_name(filename)
    rules = (
        (r"hurto.*modalidad|modalidad.*hurto", "HURTO POR MODALIDADES", "complementaria"),
        (r"hurto.*persona|persona.*hurto", "HURTO A PERSONAS", "principal"),
        (r"homicid", "HOMICIDIO", "principal"),
        (r"lesion", "LESIONES PERSONALES", "principal"),
        (r"violencia.*intrafamiliar|intrafamiliar", "VIOLENCIA INTRAFAMILIAR", "principal"),
        (r"sexual", "DELITOS SEXUALES", "principal"),
        (r"amenaza", "AMENAZAS", "principal"),
        (r"extors", "EXTORSION", "principal"),
    )
    matches = [(label, role) for pattern, label, role in rules if re.search(pattern, name)]
    if len(matches) != 1:
        raise ValueError(
            f"No se pudo clasificar de forma única la fuente {filename}: {matches}"
        )
    return matches[0]


def find_column(columns, candidates, required=True):
    normalized = {normalize_name(column): column for column in columns}
    for candidate in candidates:
        if normalize_name(candidate) in normalized:
            return normalized[normalize_name(candidate)]
    if required:
        raise KeyError(
            f"Falta una columna requerida entre {candidates}. "
            f"Disponibles: {list(columns)}"
        )
    return None


## 3. Auditoría SIEDCO antes de consolidar

Cada archivo produce un dataset filtrado independiente. Si una fuente
no contiene código oficial, `cantidad` o una fecha reconocible, se
registra como fallida y no entra al flujo. El proceso continúa para
auditar las demás fuentes y luego bloquea la fase completa.


In [ ]:
if not SIEDCO_INPUT.is_dir():
    raise FileNotFoundError(f"No existe la carpeta SIEDCO: {SIEDCO_INPUT}")

siedco_files = sorted(SIEDCO_INPUT.glob("*.csv"))
if len(siedco_files) != EXPECTED_SIEDCO_FILES:
    raise ValueError(
        f"Se esperaban {EXPECTED_SIEDCO_FILES} CSV y se encontraron "
        f"{len(siedco_files)}: {[path.name for path in siedco_files]}"
    )

audit_rows = []
for path in siedco_files:
    audit = {
        "archivo": path.name,
        "ruta_origen": str(path),
        "sha256": file_sha256(path),
        "bytes": path.stat().st_size,
        "estado": "FALLA",
    }
    try:
        data, encoding, separator = read_csv_robust(path)
        original_columns = list(data.columns)
        data.columns = [normalize_name(column) for column in data.columns]
        if len(set(data.columns)) != len(data.columns):
            raise ValueError("La normalización produjo nombres de columna duplicados.")

        source_type, source_role = infer_source_type(path.name)
        quantity_column = find_column(data.columns, ("cantidad",))
        date_column = find_column(
            data.columns,
            ("fecha_hecho", "fecha", "fecha_del_hecho", "fecha_ocurrencia"),
        )

        if "cod_muni" in data.columns:
            city_mask = normalize_code(data["cod_muni"]).eq("76001")
            code_rule = "cod_muni=76001"
        elif "codigo_dane" in data.columns:
            city_mask = normalize_code(data["codigo_dane"]).eq("76001000")
            code_rule = "codigo_dane=76001000"
        else:
            raise KeyError(
                "No existe cod_muni ni codigo_dane; no se filtra por nombre."
            )

        filtered = data.loc[city_mask].copy()
        filtered["_fecha_estandar"] = parse_dates_robust(filtered[date_column])
        filtered["_cantidad_estandar"] = pd.to_numeric(
            filtered[quantity_column]
            .astype("string")
            .str.replace(",", ".", regex=False),
            errors="coerce",
        )
        if filtered.empty:
            raise ValueError("El filtro por código oficial no devolvió filas de Cali.")
        if filtered["_cantidad_estandar"].isna().any():
            raise ValueError("Hay valores de cantidad no numéricos en filas de Cali.")
        if (filtered["_cantidad_estandar"] < 0).any():
            raise ValueError("Hay cantidades negativas en filas de Cali.")

        municipality_column = find_column(
            filtered.columns,
            ("municipio", "nombre_municipio", "ciudad", "municipio_hecho"),
            required=False,
        )
        if municipality_column:
            municipality_names = filtered[municipality_column].dropna().map(normalize_name)
            compatible = municipality_names.str.contains(
                r"(?:^|_)cali(?:$|_)|santiago_de_cali", regex=True
            )
            municipality_check = (
                f"{compatible.mean():.1%} de nombres compatibles con Cali"
                if len(compatible)
                else "Sin nombres no nulos"
            )
        else:
            municipality_check = "Columna no disponible"

        source_id = normalize_name(path.stem)
        filtered["_fuente_id"] = source_id
        filtered["_archivo_origen"] = path.name
        filtered["_tipo_fuente"] = source_type
        filtered["_rol_fuente"] = source_role
        filtered["_regla_cali"] = code_rule
        filtered["_fila_origen"] = filtered.index

        output_path = SIEDCO_LANDING / f"{source_id}.csv"
        filtered.to_csv(output_path, index=False, encoding="utf-8-sig")

        audit.update(
            {
                "estado": "OK",
                "tipo_fuente": source_type,
                "rol_fuente": source_role,
                "encoding": encoding,
                "separador": separator,
                "columnas_originales": json.dumps(original_columns, ensure_ascii=False),
                "filas_fuente_agregadas": len(data),
                "filas_cali_agregadas": len(filtered),
                "regla_cali": code_rule,
                "validacion_municipio_secundaria": municipality_check,
                "columna_fecha": date_column,
                "fechas_invalidas_cali": int(filtered["_fecha_estandar"].isna().sum()),
                "fecha_min_cali": filtered["_fecha_estandar"].min(),
                "fecha_max_cali": filtered["_fecha_estandar"].max(),
                "total_cantidad_cali": filtered["_cantidad_estandar"].sum(min_count=1),
                "salida_derivada": str(output_path),
            }
        )
    except Exception as exc:
        audit["error"] = f"{type(exc).__name__}: {exc}"
    audit_rows.append(audit)

audit_siedco = pd.DataFrame(audit_rows)
audit_siedco.to_csv(AUDIT / "inventario_siedco.csv", index=False, encoding="utf-8-sig")
display(audit_siedco)

failed_sources = audit_siedco.loc[audit_siedco["estado"] != "OK"]
if not failed_sources.empty:
    raise RuntimeError(
        "La fase queda bloqueada por fuentes SIEDCO fallidas. "
        "Revise audit/inventario_siedco.csv."
    )
if (audit_siedco["rol_fuente"] == "complementaria").sum() != 1:
    raise ValueError(
        "Debe existir exactamente una fuente complementaria Hurto por Modalidades."
    )


## 4. Auditoría defensiva de población DANE

Se buscan Excel cuyo nombre indique DANE, población o proyecciones.
Cada hoja se prueba con encabezados en las primeras diez filas. Solo se
conserva una hoja si presenta un código municipal completo y contiene
Cali por `76001` o `76001000`. Todavía no se calculan tasas.


In [ ]:
DANE_FILE_HINTS = ("dane", "poblacion", "proyeccion")
DANE_CODE_COLUMNS = (
    "codigo_dane", "cod_dane", "cod_muni", "codigo_municipio",
    "cod_municipio", "cod_mpio", "dpmp",
)

dane_files = sorted(
    path
    for pattern in ("*.xlsx", "*.xls")
    for path in DATAV3_ROOT.rglob(pattern)
    if PROJECT_OUTPUT not in path.parents
    and not path.name.startswith("~$")
    and any(hint in normalize_name(path.name) for hint in DANE_FILE_HINTS)
)
if not dane_files:
    raise FileNotFoundError(
        "No se identificaron Excel DANE en MyDrive/datav3 por nombre."
    )

dane_audit_rows = []
valid_dane_outputs = []
for path in dane_files:
    workbook_found = False
    try:
        sheet_names = pd.ExcelFile(path).sheet_names
    except Exception as exc:
        dane_audit_rows.append(
            {
                "archivo": str(path.relative_to(DATAV3_ROOT)),
                "estado": "FALLA",
                "error": f"No se pudo abrir el libro: {exc}",
            }
        )
        continue

    for sheet in sheet_names:
        accepted = None
        accepted_header = None
        accepted_code = None
        for header_row in range(10):
            try:
                candidate = pd.read_excel(path, sheet_name=sheet, header=header_row)
                candidate.columns = [normalize_name(column) for column in candidate.columns]
                code_column = find_column(
                    candidate.columns, DANE_CODE_COLUMNS, required=False
                )
                if code_column:
                    codes = normalize_code(candidate[code_column])
                    if codes.isin(["76001", "76001000"]).any():
                        accepted = candidate
                        accepted_header = header_row
                        accepted_code = code_column
                        break
            except Exception:
                continue

        if accepted is None:
            continue

        workbook_found = True
        codes = normalize_code(accepted[accepted_code])
        cali_dane = accepted.loc[codes.isin(["76001", "76001000"])].copy()
        cali_dane["_archivo_origen"] = path.name
        cali_dane["_hoja_origen"] = str(sheet)
        cali_dane["_codigo_cali_usado"] = accepted_code
        output_name = (
            f"{normalize_name(path.stem)}__{normalize_name(sheet)}.csv"
        )
        output_path = DANE_LANDING / output_name
        cali_dane.to_csv(output_path, index=False, encoding="utf-8-sig")
        valid_dane_outputs.append(output_path)
        dane_audit_rows.append(
            {
                "archivo": str(path.relative_to(DATAV3_ROOT)),
                "sha256": file_sha256(path),
                "hoja": sheet,
                "fila_encabezado": accepted_header,
                "estado": "OK",
                "columna_codigo": accepted_code,
                "filas_cali": len(cali_dane),
                "salida_derivada": str(output_path),
            }
        )
    if not workbook_found:
        dane_audit_rows.append(
            {
                "archivo": str(path.relative_to(DATAV3_ROOT)),
                "estado": "FALLA",
                "error": "Ninguna hoja incluyó código municipal completo para Cali.",
            }
        )

audit_dane = pd.DataFrame(dane_audit_rows)
audit_dane.to_csv(AUDIT / "inventario_dane.csv", index=False, encoding="utf-8-sig")
display(audit_dane)
if not valid_dane_outputs:
    raise RuntimeError(
        "No hay una fuente DANE válida. Revise audit/inventario_dane.csv."
    )


## 5. Registro de ejecución y cierre

**Comprobación esperada:** ocho fuentes SIEDCO en estado `OK`, una sola
fuente marcada como complementaria y al menos una hoja DANE válida.

**Decisión:** si cualquier fuente falla, no se consolida. Corregir la
ruta, el nombre o el esquema en origen y ejecutar nuevamente. Los
archivos originales nunca son sobrescritos.


In [ ]:
run_manifest = {
    "version_salida": "eda_01_05_v1",
    "entrada_siedco": str(SIEDCO_INPUT),
    "busqueda_dane": str(DATAV3_ROOT),
    "salida_proyecto": str(PROJECT_OUTPUT),
    "fuentes_siedco_ok": int((audit_siedco["estado"] == "OK").sum()),
    "hojas_dane_ok": int((audit_dane["estado"] == "OK").sum()),
    "periodo_eda": [PERIODO_INICIO, PERIODO_FIN],
    "nota_grano": "Cada fila SIEDCO es un agregado; los casos suman cantidad.",
}
(AUDIT / "manifest_ejecucion.json").write_text(
    json.dumps(run_manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps(run_manifest, ensure_ascii=False, indent=2))
